In [1]:
# Q2: CIFAR-10 Colorization with Encoder-Decoder (per-pixel CE)

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Dataset
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

CENTROIDS_PATH = '/kaggle/input/color-centroid/color_centroids.npy'
centroids = np.load(CENTROIDS_PATH)  # shape (24, 3), values assumed in [0,1]
centroids_t = torch.tensor(centroids, dtype=torch.float32)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Using device: cuda


In [2]:
# Dataset: CIFAR-10 grayscale input + 24-class labels via color centroids

class CIFARColorizationDataset(Dataset):
    def __init__(self, split='train', root='./data', download=True, augment=False):
        assert split in ['train', 'val', 'test']
        base_train = torchvision.datasets.CIFAR10(root=root, train=True, download=download)
        base_test = torchvision.datasets.CIFAR10(root=root, train=False, download=download)
        if split in ['train', 'val']:
            # 45k train / 5k val
            train_size = 45000
            val_size = len(base_train) - train_size
            train_subset, val_subset = random_split(
                base_train, [train_size, val_size], generator=torch.Generator().manual_seed(42)
            )
            self.base = train_subset if split == 'train' else val_subset
        else:
            self.base = base_test
        self.augment = augment
        self.flip = T.RandomHorizontalFlip(p=0.5) if augment else None
        self.to_tensor = T.ToTensor()  # converts pil -> [0,1]

    def __len__(self):
        return len(self.base)

    def _rgb_to_labels(self, rgb):
        # rgb: Tensor [3, H, W] in [0,1]
        c = centroids_t  # move centroids to CPU for computation
        H, W = rgb.shape[1], rgb.shape[2]
        flat = rgb.view(3, -1).transpose(0, 1)  # [H*W, 3]
        # compute squared distances to centroids
        d2 = torch.cdist(flat.unsqueeze(0), c.unsqueeze(0)).squeeze(0)
        labels = torch.argmin(d2, dim=1).view(H, W).long()
        return labels


    def __getitem__(self, idx):
        img, _ = self.base[idx]  # PIL image, ignore class label
        if self.augment:
            img = self.flip(img)
        rgb = self.to_tensor(img)  # [3,32,32]
        # grayscale single-channel input
        gray = 0.2989 * rgb[0:1] + 0.5870 * rgb[1:2] + 0.1140 * rgb[2:3]  # [1,32,32]
        labels = self._rgb_to_labels(rgb)  # [32,32]
        return gray, labels, rgb


def load_cifar_colorization(batch_size=128):
    train_ds = CIFARColorizationDataset(split='train', augment=True)
    val_ds = CIFARColorizationDataset(split='val', augment=False)
    test_ds = CIFARColorizationDataset(split='test', augment=False)
    common = dict(num_workers=2, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, **common)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, **common)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, **common)
    return train_loader, val_loader, test_loader

print('Loading CIFAR-10 colorization dataset...')
train_loader, val_loader, test_loader = load_cifar_colorization(batch_size=128)
xb, yb, rb = next(iter(train_loader))
print('Batch:', xb.shape, yb.shape, rb.shape)


Loading CIFAR-10 colorization dataset...


100%|██████████| 170M/170M [00:05<00:00, 33.3MB/s]


Batch: torch.Size([128, 1, 32, 32]) torch.Size([128, 32, 32]) torch.Size([128, 3, 32, 32])


In [3]:
# Model: Encoder-Decoder with 3 down + 3 up + classifier

class ColorizeED(nn.Module):
    def __init__(self, NIC=1, NF=64, NC=24, k_enc=3):
        super().__init__()
        p = k_enc // 2
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(NIC, NF, k_enc, padding=p), nn.BatchNorm2d(NF), nn.ReLU(inplace=True), nn.MaxPool2d(2)
        )  # [B, NF, 16, 16]
        self.enc2 = nn.Sequential(
            nn.Conv2d(NF, 2*NF, k_enc, padding=p), nn.BatchNorm2d(2*NF), nn.ReLU(inplace=True), nn.MaxPool2d(2)
        )  # [B, 2*NF, 8, 8]
        self.enc3 = nn.Sequential(
            nn.Conv2d(2*NF, 4*NF, k_enc, padding=p), nn.BatchNorm2d(4*NF), nn.ReLU(inplace=True), nn.MaxPool2d(2)
        )  # [B, 4*NF, 4, 4]
        # Decoder (keep kernel=4, stride=2 for learned upsampling)
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(4*NF, 2*NF, 4, stride=2, padding=1), nn.BatchNorm2d(2*NF), nn.ReLU(inplace=True)
        )  # [B, 2*NF, 8, 8]
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(2*NF, NF, 4, stride=2, padding=1), nn.BatchNorm2d(NF), nn.ReLU(inplace=True)
        )  # [B, NF, 16, 16]
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(NF, NC, 4, stride=2, padding=1), nn.BatchNorm2d(NC), nn.ReLU(inplace=True)
        )  # [B, NC, 32, 32]
        # Classifier to logits [B, NC, 32, 32]
        self.cls = nn.Conv2d(NC, NC, 1)

    def forward(self, x):
        h1 = self.enc1(x)
        h2 = self.enc2(h1)
        h3 = self.enc3(h2)
        u1 = self.dec1(h3)
        u2 = self.dec2(u1)
        u3 = self.dec3(u2)
        logits = self.cls(u3)
        return logits

model = ColorizeED().to(device)
print(model.__class__.__name__, 'params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


ColorizeED params: 1051744


In [4]:
# Training: per-pixel CE, wandb logging, save best (val loss)

def logits_to_rgb(logits):
    # logits: [B, NC, H, W] -> [B, 3, H, W] via centroids argmax
    with torch.no_grad():
        pred = logits.argmax(dim=1)  # [B,H,W]
        B, H, W = pred.size(0), pred.size(1), pred.size(2)
        flat = pred.view(B, -1)  # [B, H*W]
        palette = centroids_t  # [NC,3]
        rgb = palette[flat]  # [B, H*W, 3]
        rgb = rgb.view(B, H, W, 3).permute(0, 3, 1, 2).contiguous()
        return rgb.clamp(0, 1)


def visualize_batch(gray, labels, logits, max_images=10):
    rgb_pred = logits_to_rgb(logits).cpu()
    gray = gray.cpu()
    labels = labels.cpu()
    # ground-truth rgb from labels
    with torch.no_grad():
        B, H, W = labels.size(0), labels.size(1), labels.size(2)
        flat = labels.view(B, -1)
        gt_rgb = centroids_t[flat.to(centroids_t.device)].view(B, H, W, 3).permute(0, 3, 1, 2).cpu()
    n = min(max_images, gray.size(0))
    fig, axes = plt.subplots(n, 3, figsize=(6, 2*n))
    if n == 1:
        axes = axes.unsqueeze(0)
    for i in range(n):
        axes[i,0].imshow(gray[i,0], cmap='gray'); axes[i,0].set_title('Gray'); axes[i,0].axis('off')
        axes[i,1].imshow(rgb_pred[i].permute(1,2,0)); axes[i,1].set_title('Pred'); axes[i,1].axis('off')
        axes[i,2].imshow(gt_rgb[i].permute(1,2,0)); axes[i,2].set_title('GT'); axes[i,2].axis('off')
    plt.tight_layout()
    return fig


def train_colorization(config):
    run = wandb.init(project=config.get('project','cifar10-colorization'), name=config.get('run_name'), config=config, reinit=True)
    train_loader, val_loader, test_loader = load_cifar_colorization(batch_size=config['batch_size'])
    model = ColorizeED(NIC=1, NF=config['NF'], NC=24, k_enc=config.get('k', 3)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=config['lr']) if config['optimizer']=='adam' else torch.optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9)
    best_val = float('inf'); best_path = None
    ce_loss = nn.CrossEntropyLoss()

    for epoch in range(1, config['epochs']+1):
        model.train()
        tot, cnt = 0.0, 0
        for gray, labels, _ in train_loader:
            gray, labels = gray.to(device), labels.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(gray)
            loss = ce_loss(logits, labels)
            loss.backward(); opt.step()
            b = gray.size(0); tot += loss.item()*b; cnt += b
        train_loss = tot/cnt

        model.eval()
        vtot, vcnt = 0.0, 0
        with torch.no_grad():
            for gray, labels, _ in val_loader:
                gray, labels = gray.to(device), labels.to(device)
                logits = model(gray)
                loss = ce_loss(logits, labels)
                b = gray.size(0); vtot += loss.item()*b; vcnt += b
        val_loss = vtot/vcnt

        wandb.log({'epoch': epoch, 'train/loss': train_loss, 'val/loss': val_loss}, step=epoch)

        if val_loss < best_val:
            best_val = val_loss
            best_path = f"checkpoints/color_best_{wandb.run.id}.pt"
            os.makedirs('checkpoints', exist_ok=True)
            torch.save({'model': model.state_dict(), 'config': dict(wandb.config)}, best_path)
            wandb.summary['best_val_loss'] = best_val
            wandb.summary['best_ckpt'] = best_path

        # occasional visualization
        if epoch % max(1, config['epochs']//5) == 0:
            gray, labels, _ = next(iter(val_loader))
            gray, labels = gray.to(device), labels.to(device)
            with torch.no_grad():
                logits = model(gray)
            fig = visualize_batch(gray.cpu(), labels.cpu(), logits.cpu(), max_images=5)
            wandb.log({'val/examples': wandb.Image(fig)}, step=epoch)
            plt.close(fig)

    # test eval (loss + examples)
    model.eval()
    ttot, tcnt = 0.0, 0
    with torch.no_grad():
        for gray, labels, _ in test_loader:
            gray, labels = gray.to(device), labels.to(device)
            logits = model(gray)
            loss = ce_loss(logits, labels)
            b = gray.size(0); ttot += loss.item()*b; tcnt += b
    test_loss = ttot/tcnt
    wandb.summary['test/loss'] = test_loss

    # 10 example colorizations
    gray, labels, _ = next(iter(test_loader))
    gray, labels = gray.to(device), labels.to(device)
    with torch.no_grad():
        logits = model(gray)
    fig = visualize_batch(gray.cpu(), labels.cpu(), logits.cpu(), max_images=10)
    wandb.log({'test/examples': wandb.Image(fig)})
    plt.close(fig)

    wandb.finish()
    return {'best_val_loss': best_val, 'test_loss': test_loss, 'best_ckpt': best_path}


In [5]:
# wandb sweep: 20+ configurations over lr, batch_size, NF, kernel size, optimizer

sweep_config = {
    'method': 'random',
    'metric': {'name': 'val/loss', 'goal': 'minimize'},
    'parameters': {
        'lr': {'values': [1e-4, 3e-4, 1e-3, 3e-3]},
        'batch_size': {'values': [32, 64, 128]},
        'NF': {'values': [8, 16, 32]},
        'k': {'values': [3, 5]},
        'optimizer': {'values': ['sgd', 'adam']},
        'epochs': {'value': 25},
        'project': {'value': 'cifar10-colorization'},
    }
}


def sweep_train():
    # Initialize wandb - this must be called before accessing wandb.config
    # When using wandb.agent(), init must be called explicitly in the function
    wandb.init(project='cifar10-colorization')
    cfg = dict(wandb.config)
    # Provide run_name to help identify
    cfg['run_name'] = f"nf{cfg['NF']}_k{cfg['k']}_bs{cfg['batch_size']}_{cfg['optimizer']}_lr{cfg['lr']}"
    train_colorization(cfg)

# To launch the sweep:
# 1) Login to wandb: wandb.login()
# 2) Create sweep:
#    sweep_id = wandb.sweep(sweep_config, project='cifar10-colorization')
# 3) Run the agent (20+ runs):
#    wandb.agent(sweep_id, function=sweep_train, count=20)

print('Sweep config ready. Use wandb.sweep(...) and wandb.agent(..., count=20).')


Sweep config ready. Use wandb.sweep(...) and wandb.agent(..., count=20).


### Assumptions
- Encoder conv blocks: Conv2d(kernel=3, stride=1, padding=1) → BN → ReLU → MaxPool2d(kernel=2, stride=2)
- Decoder blocks: ConvTranspose2d(kernel=4, stride=2, padding=1) → BN → ReLU
- Final classifier: Conv2d(kernel=1, stride=1, padding=0)
- Ignore biases and BN params; count only conv/convT weights.
- Base spatial size: 32×32. Doubled case: 64×64.
- Channel plan per spec:
  - After enc1: [NF, 16,16], enc2: [2NF, 8,8], enc3: [4NF, 4,4]
  - Dec1: [2NF, 8,8], Dec2: [NF, 16,16], Dec3: [NC, 32,32], Classifier: [NC, 32,32]

### 1) Number of weights (independent of spatial size)
- Conv1 (NIC→NF, k=3): 9·NIC·NF
- Conv2 (NF→2NF, k=3): 9·NF·(2NF) = 18·NF²
- Conv3 (2NF→4NF, k=3): 9·(2NF)·(4NF) = 72·NF²
- Deconv1 (4NF→2NF, k=4): 16·(4NF)·(2NF) = 128·NF²
- Deconv2 (2NF→NF, k=4): 16·(2NF)·NF = 32·NF²
- Deconv3 (NF→NC, k=4): 16·NF·NC
- Classifier (NC→NC, k=1): NC²

Total weights:
- 9·NIC·NF + (18+72+128+32)·NF² + 16·NF·NC + NC²
- = 9·NIC·NF + 250·NF² + 16·NF·NC + NC²

### 2) Number of outputs (total activation elements across blocks, single input)
Base 32×32:
- After enc1: NF·16·16 = 256·NF
- After enc2: 2NF·8·8 = 128·NF
- After enc3: 4NF·4·4 = 64·NF
- After dec1: 2NF·8·8 = 128·NF
- After dec2: NF·16·16 = 256·NF
- After dec3: NC·32·32 = 1024·NC
- After classifier: NC·32·32 = 1024·NC

Total outputs (32×32):
- (256+128+64+128+256)·NF + (1024+1024)·NC
- = 832·NF + 2048·NC

Doubled 64×64:
- After enc1: NF·32·32 = 1024·NF
- After enc2: 2NF·16·16 = 512·NF
- After enc3: 4NF·8·8 = 256·NF
- After dec1: 2NF·16·16 = 512·NF
- After dec2: NF·32·32 = 1024·NF
- After dec3: NC·64·64 = 4096·NC
- After classifier: NC·64·64 = 4096·NC

Total outputs (64×64):
- (1024+512+256+512+1024)·NF + (4096+4096)·NC
- = 3328·NF + 8192·NC

### 3) Number of connections
Definition: for Conv2d, each weight is applied at every output spatial position → connections per weight = Hout·Wout. For ConvTranspose2d, each weight is applied at every input spatial position → connections per weight = Hin·Win.

Base 32×32:
- Conv1 (Hout=32): (9·NIC·NF)·(32·32) = 9216·NIC·NF
- Conv2 (Hout=16): (18·NF²)·(16·16) = 4608·NF²
- Conv3 (Hout=8): (72·NF²)·(8·8) = 4608·NF²
- Deconv1 (Hin=4): (128·NF²)·(4·4) = 2048·NF²
- Deconv2 (Hin=8): (32·NF²)·(8·8) = 2048·NF²
- Deconv3 (Hin=16): (16·NF·NC)·(16·16) = 4096·NF·NC
- Classifier (Hout=32): (NC²)·(32·32) = 1024·NC²

Total connections (32×32):
- 9216·NIC·NF + (4608+4608+2048+2048)·NF² + 4096·NF·NC + 1024·NC²
- = 9216·NIC·NF + 13312·NF² + 4096·NF·NC + 1024·NC²

Doubled 64×64:
- Conv1 (Hout=64): (9·NIC·NF)·(64·64) = 36864·NIC·NF
- Conv2 (Hout=32): (18·NF²)·(32·32) = 18432·NF²
- Conv3 (Hout=16): (72·NF²)·(16·16) = 18432·NF²
- Deconv1 (Hin=8): (128·NF²)·(8·8) = 8192·NF²
- Deconv2 (Hin=16): (32·NF²)·(16·16) = 8192·NF²
- Deconv3 (Hin=32): (16·NF·NC)·(32·32) = 16384·NF·NC
- Classifier (Hout=64): (NC²)·(64·64) = 4096·NC²

Total connections (64×64):
- 36864·NIC·NF + (18432+18432+8192+8192)·NF² + 16384·NF·NC + 4096·NC²
- = 36864·NIC·NF + 53248·NF² + 16384·NF·NC + 4096·NC²

These expressions match the specified baseline (3 down/3 up + 1×1 classifier) and scale cleanly with spatial size.

In [6]:
print("hi")

hi


In [7]:
wandb.login(key="57d7791c310a4977e39e823b8d7405892f51e8aa")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: malavathulakv (malavathulakv-iiit-hyderabad) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
import wandb
wandb.login()

True

In [9]:
import wandb
wandb.login()
sweep_id = wandb.sweep(sweep_config, project='cifar10-colorization')
wandb.agent(sweep_id, function=sweep_train, count=20)


Create sweep with ID: ic89yo4y
Sweep URL: https://wandb.ai/malavathulakv-iiit-hyderabad/cifar10-colorization/sweeps/ic89yo4y


wandb: Agent Starting Run: kox3olvz with config:
wandb: 	NF: 16
wandb: 	batch_size: 64
wandb: 	epochs: 25
wandb: 	k: 3
wandb: 	lr: 0.0001
wandb: 	optimizer: adam
wandb: 	project: cifar10-colorization
wandb: WARNING Ignoring project 'cifar10-colorization' when running a sweep.
wandb: Tracking run with wandb version 0.21.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251101_194935-kox3olvz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run grateful-sweep-1
wandb: ⭐️ View project at https://wandb.ai/malavathulakv-iiit-hyderabad/cifar10-colorization
wandb: 🧹 View sweep at https://wandb.ai/malavathulakv-iiit-hyderabad/cifar10-colorization/sweeps/ic89yo4y
wandb: 🚀 View run at https://wandb.ai/malavathulakv-iiit-hyderabad/cifar10-colorization/runs/kox3olvz
wandb: WARNING Ignoring project 'cifar10-colorization' when running a sweep.
wandb: Finishing previous runs because reinit is set to True.
wandb:                                                            